# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
I. J. M. Crossfield  ->  I. J. M. Crossfield  |  ['I. J. M. Crossfield']
J. Li  ->  J. Li  |  ['J. Li']
J. Li  ->  J. Li  |  ['J. Li']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
A. Pillepich  ->  A. Pillepich  |  ['A. Pillepich']
S. Shahaf  ->  S. Shahaf  |  ['S. Shahaf']
K. El-Badry  ->  K. El-Badry  |  ['K. El-Badry']
H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']
R. Andrae  ->  R. Andrae  |  ['R. Andrae']
Arxiv has 113 new papers today
          8 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/8 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.19899


extracting tarball to tmp_2611.19899... done.
Retrieving document from  https://arxiv.org/e-print/2611.20166


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.19899/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'contributions' from 'tmp_2611.19899/contributions.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.19899/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-package

extracting tarball to tmp_2611.20166...

 done.


... (14 outputs left out)

A. Pillepich  ->  A. Pillepich  |  ['A. Pillepich']


Retrieving document from  https://arxiv.org/e-print/2611.21510


extracting tarball to tmp_2611.21510... done.


Found 36 bibliographic references in tmp_2611.21510/main.bbl.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20633-b31b1b.svg)](https://arxiv.org/abs/2611.20633) | **Model-Driven Stellar Spectroscopy with the Total Differential Spectral Model (TDSM): Framework and Its Application to 11 Million LAMOST Low-Resolution Spectra**  |
|| L. Mou, et al. -- incl., <mark>M. Zhang</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| **|
|**Abstract**|            Deriving stellar astrophysical parameters from large-scale spectroscopic surveys requires accurate modeling of spectra across a wide wavelength coverage and a broad range of stellar parameter space, which remains a great challenge. In this work, we present \texttt{TDSM}, a physically motivated neural network architecture that self-consistently emulates both the flux and its first-order derivative of the ab initio stellar spectra. The method enables accurate and precise emulation of Kurucz synthetic spectra with severely blended features at $R \simeq 2000$ in a high-dimensional parameter space, achieving a typical pixel-wise flux precision of 0.05\% in the wavelength range $3700<\lambda<9000$Å over the parameter range of $3000 < T_{\rm eff} < 45,000$~K, $-1 < \log g < 5$, and $-4.0 < \mathrm{[Fe/H]} < 0.5$. The model is applied to LAMOST DR10 to derive stellar parameters for 11 million spectra spanning the full model parameter range. We conduct detailed refinement and calibration efforts to mitigate systematic errors, which are revealed to be significant and complex due to imperfections in the ab initio model atmosphere. The resultant catalog provides eight stellar labels -- $T_{\rm eff}$, $\log g$, $v_{\rm mic}$, [Fe/H], [$\alpha$/Fe], [C/Fe], [N/Fe], and [Mg/Fe] -- for 8 million unique stars (11 million spectra including repeat observations) across the full parameter space. This adds a valuable dataset for studies of Galactic structure and archaeology, stellar populations, and related areas of astrophysics.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20785-b31b1b.svg)](https://arxiv.org/abs/2611.20785) | **Validation of Ariel candidates orbiting K-M stars**  |
|| G. Piccinini, et al. -- incl., <mark>I. J. M. Crossfield</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *Accepted for publication on Astronomy and Astrophysics*|
|**Abstract**|            One of the key goals of planetary population studies is to achieve broad coverage of the parameter space, which is not yet fully represented by confirmed planets. This limitation can be mitigated by including planets that have been statistically validated through a vetting procedure, even if they are not fully confirmed. In preparation for the Ariel mission, a Mission Candidate Sample has been assembled, comprising both confirmed planets and several unvalidated candidates. In this context, we apply a vetting analysis to a selected subsample of these candidates. Our methodology is based on the TRICERATOPS tool, aimed at confirming the presence of transiting planets or ruling out false-positive scenarios. We applied TRICERATOPS to a sample of 318 targets drawn from the last three Ariel TESS Planet Candidates catalogues. The analysis includes evaluation of statistical probabilities, as well as checks on stellar binarity indicators such as the Renormalized Unit Weight Error (RUWE) and the IPD_frac_multi_peak values from Gaia Data Release 3. Our analysis validated 56 candidates as planets, identified 29 false positives, and classified 35 as likely planets. These results are intended to support the Ariel mission by providing a probabilistic priority ranking to guide follow-up observations and optimise targets for atmospheric characterisation studies.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20884-b31b1b.svg)](https://arxiv.org/abs/2611.20884) | **Orbital decomposition and evolution of LEGA-C-like galaxies in IllustrisTNG from z = 1 to z = 0**  |
|| P. Kumar, et al. -- incl., <mark>A. Pillepich</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *Accepted for publication in A&A on 27/10/2026*|
|**Abstract**|            Stellar orbits encode the dynamical history of galaxies and provide a physically motivated link between galaxy structures and assembly processes. Although orbit-based decompositions are well established in the Local Universe, their evolution across cosmic time, especially at higher redshifts, remains poorly explored. We investigate the orbital structure of \texttt{LEGA-C-like} galaxies in the \texttt{TNG100} simulation from $z=1$ to $z=0$ to identify how stellar mass, star formation activity, environment, merger history, gas content, and ex situ stellar mass growth shape their orbital transformation. We decomposed the stellar orbital structure of simulated galaxies into cold, warm, hot, and counter-rotating (CR) components using time-averaged orbital circularities. We find a strong mass dependence of orbital structure at $z=0$ as well as at $z=1$. Galaxies below $\log(M_\star/M_\odot)\sim11.2$ have a combined warm and cold orbital fraction of $\sim70\%$, whereas more massive systems have a hot-orbit fraction of $\sim60\%$. These trends are consistent with assembly pathways driven predominantly by wet and dry major mergers, respectively. Tracking the same galaxies to $z=0$ reveals a systematic, but mass-dependent, evolution toward lower rotational support: At both low and high stellar masses, the hot-orbit fraction increases, whereas the relative change is greater in low-mass galaxies. The survival of cold structure depends strongly on the star formation state at $z=1$; star-forming progenitors retain substantially larger cold fractions than the quiescent progenitors. At both redshifts, the orbital fractions show a rise (decline) in hot and CR (cold and warm) orbits with the ex situ stellar mass fraction at a fixed stellar mass. These results provide the first evolutionary framework for interpreting orbit-based decompositions of galaxies at the \texttt{LEGA-C} epoch.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.21510-b31b1b.svg)](https://arxiv.org/abs/2611.21510) | **Gaia as a 0.1 arcsec point-source survey across the sky: dynamical scene reconstruction from one-dimensional scans**  |
|| <mark>S. Shahaf</mark>, et al. -- incl., <mark>K. El-Badry</mark>, <mark>H.-W. Rix</mark>, <mark>R. Andrae</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *Submitted to A&A. 17 pages, 8 figures. Comments are welcome. Code available at this https URL*|
|**Abstract**|            Gaia has observed every detected source on the sky 50 to 100 times, each time through a window of about 1 by 2 arcseconds at a different scan angle. The resulting one-dimensional flux profiles to be released in Gaia DR4 encode the two-dimensional arrangement of nearby point sources at ~0.1 arcsecond resolution. Repeated observations also constrain their relative motion. We present a method to model the scene with a mixture of moving point sources. Complementing Gaia DPAC's source-environment reconstruction, our approach focuses on close, blended companions and models their relative motion. We first fit each window with a few point sources and obtain their along-scan positions. We then triangulate these positions from different scan angles to identify candidate sources on the sky. A joint forward model of all windows measures positions, fluxes, relative proper motions, and parallax differences, even for blended components. In simulations, we recover companions at separations smaller than the instrumental profile width. For a moving companion at ~75 mas separation with a flux ratio of ~0.2, we obtain a separation precision of ~0.1 mas and a relative proper-motion precision of ~0.1 mas/yr per component, even in a scene contaminated by a third source. These precisions assume a known instrumental profile and independent Gaussian noise. We also recover differential parallax for a chance-aligned source. Modeling nearby contaminants prevents significant biases. With a Gaia-like instrumental profile, sampling and scan law, we show a sensitivity of ~5 mag in contrast at separations of 50-150 mas, and we discuss its dependence on the color term of Gaia's line-spread function. The method could extend the census of close companions and provide relative astrometry, and so dynamical information, along short orbital arcs. External measurements could also guide retrospective searches in Gaia's windows.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20669-b31b1b.svg)](https://arxiv.org/abs/2611.20669) | **Long-Term Radio Evolution of the Type Ia Supernova SN 2020eyj: Evidence for a Dense Helium-rich Circumstellar Shell**  |
|| Z.-W. Liu, et al. -- incl., <mark>J. Li</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *17 pages, 10 figures, 3 tables, submitted to AAS*|
|**Abstract**|            The nature of progenitor systems of Type Ia supernovae (SNe Ia) and their explosion mechanism remain controversial. Radio observations provide a powerful probe of the interaction between SN ejecta and circumstellar material (CSM), offering important constraints on the mass-loss histories of their progenitor systems. We present extensive Jansky Very Large Array (JVLA) observations of SN 2020eyj at 1.5-10 GHz, tracing its radio emission to approximately 1300 days after discovery and revealing, for the first time, the long-term radio evolution of an SN Ia. The radio light curves of SN 2020eyj exhibit a relatively flat evolution at early epochs, followed by a gradual long-term decline, while its radio spectra show a power-law behavior with a low-frequency turnover. These characteristics appear to be consistent with synchrotron emission associated with shock interaction. Modeling the radio light curves favors a thick circumstellar shell with an inferred mass of approximately 0.3-0.4 solar masses over the simple wind-like configurations. Combined with the optical spectral signatures of SN 2020eyj, our radio observations provide strong evidence for substantial helium-rich mass loss from the progenitor system prior to explosion. The inferred CSM properties pose significant challenges to existing SN Ia progenitor models, although the ejection of a common envelope shortly before explosion may offer a plausible explanation.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: '69117' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.19899-b31b1b.svg)](https://arxiv.org/abs/2611.19899) | **Towards the LSST DESC pipeline for cosmology with galaxy clusters -- design and validation on simulations**  |
|| E. J. Barroso, et al. -- incl., <mark>Y. Zhang</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *23 pages, 14 figures, 2 tables*|
|**Abstract**|            We present the current implementation of an automated framework developed within the Legacy Survey of Space and Time Dark Energy Science Collaboration (LSST DESC) for galaxy cluster cosmology, in preparation for the first LSST data releases of the NSF-DOE Vera C. Rubin Observatory. The cluster cosmology pipeline integrates DESC-developed tools dedicated to galaxy/cluster catalog ingestion, cluster number count and weak lensing measurements, modeling, and parameter inference through posterior sampling, enabling reproducible end-to-end cluster cosmology analyses. We test and validate the pipeline using synthetic mock datasets and the LSST DESC cosmoDC2 simulation, performing a joint analysis of redMaPPer cluster abundance and stacked weak lensing profiles measured in bins of redshift and richness. We constrain two cosmological parameters, $\Omega_c$ and $\sigma_8$, along with six cluster mass-observable relation parameters. Our baseline analysis recovers the fiducial cosmoDC2 values of $\Omega_c$ and $\sigma_8$ within $0.51\sigma$ and $0.34\sigma$, respectively. We also showcase the flexibility of the overall pipeline by assessing the impact of fiducial cosmology and alternative modeling prescriptions on the parameter constraints. Lastly, we also discuss the current limitations of the pipeline and the developments needed for LSST data.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20828-b31b1b.svg)](https://arxiv.org/abs/2611.20828) | **LAMOST III Low-resolution Spectroscopic Survey: Overview and Target Selection**  |
|| S. Xu, et al. -- incl., <mark>J. Li</mark>, <mark>J. Li</mark>, <mark>Y. Wang</mark>, <mark>H. Zhang</mark>, <mark>H. Zhang</mark>, <mark>H. Zhang</mark>, <mark>M. Zhang</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| *24 pages, 14 figures, 2 tables. Accepted by SCIENCE CHINA: Physics, Mechanics & Astronomy*|
|**Abstract**|            We present an overview of the LAMOST III Low-resolution Spectroscopic Survey (LRS), including its scientific goals, target selection strategy, and current observing status. Targets in the main survey are selected via weighted random sampling, using the inverse local density in the Hertzsprung-Russell (HR) diagram and in the G-magnitude distribution, with additional enhancement in selected regions of the HR diagram. The main survey is complemented by 35 dedicated add-on catalogs (including very metal-poor star candidates, halo stars, hot/young stars, wide binaries, extragalactic sources and so on) and a time-domain program providing multi-epoch spectroscopy of binaries and other peculiar systems. LAMOST III LRS is designed to map the structure and evolution of the Galactic disk, reconstruct the assembly and early chemical-enrichment history of the Galactic halo, improve stellar-parameter calibration, and characterize rare and variable objects. By June 2025, the LAMOST III LRS had obtained 1.53 million low-resolution spectra from 658 plates, corresponding to 1,264,052 unique sources. Relative to LAMOST I+II, the current sample is enriched in O/B stars, subgiants, main-sequence turn-off stars, metal-poor stars, and other previously underrepresented populations. In particular, the fractions of metal-poor and very metal-poor stars increase from 3.8% and 0.4% in LAMOST I+II to 9.0% and 1.1% in LAMOST III LRS, respectively, demonstrating the improved efficiency of the revised target-selection strategy for Galactic archaeology and stellar astrophysics.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.20166-b31b1b.svg)](https://arxiv.org/abs/2611.20166) | **Euclid Data Release 1 (DR1). Double-source-plane lens candidates**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-17*|
|*Comments*| **|
|**Abstract**|            Strong gravitational-lensing systems with multiple source planes are powerful tools for probing the stellar and dark matter content of galaxies. Lens modelling can measure the relative Einstein radii of the two source planes, which encode a ratio of angular-diameter distances determined by the lens and source redshifts and the dark energy equation of state, independently of the Hubble constant. Galaxy-scale double-source-plane lenses (DSPLs) are extremely rare. We present a sample of 84 galaxy-scale DSPL candidates in Euclid Data Release 1 (DR1), of which 80 are newly identified. These systems were selected using machine-learning lens-finding models followed by visual inspection by citizen scientists and experts. We then used lens modelling to test the DSPL interpretation. Among our lens candidates, 11 form the 'Gold' sample: each shows two sets of multiple images at different radii, and the observed configuration is reproduced by a DSPL lens model. The remaining 73 'Silver' candidates have at least two lensed arcs from visual inspection, but either lack multiple images for at least one arc, have a best-fitting DSPL distance ratio close to the single-source-plane limit, remain compatible with a single-plane model with multiple sources, or have not yet been modelled. Even if all 84 candidates are DSPLs, the sample falls below the approximately 90 systems expected from extrapolating the Euclid Quick Data Release 1 (Q1) yield and the approximately 200 obtained by scaling the LensPop forecast to the DR1 area, indicating that many DSPLs remain undetected in DR1. Spectroscopy or higher-resolution imaging is required before we can be confident that the Silver candidates are DSPLs.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.20633/./calibration.png', 'tmp_2611.20633/./example+loss.png', 'tmp_2611.20633/./self_validation.png']
copying  tmp_2611.20633/./calibration.png to _build/html/
copying  tmp_2611.20633/./example+loss.png to _build/html/
copying  tmp_2611.20633/./self_validation.png to _build/html/
exported in  _build/html/2611.20633.md
    + _build/html/tmp_2611.20633/./calibration.png
    + _build/html/tmp_2611.20633/./example+loss.png
    + _build/html/tmp_2611.20633/./self_validation.png
found figures ['tmp_2611.20785/./Figures/radiustemperature.png', 'tmp_2611.20785/./Figures/Heatmap_TPC_Results_K.png', 'tmp_2611.20785/./Figures/Heatmap_TPC_Results_M.png']
copying  tmp_2611.20785/./Figures/radiustemperature.png to _build/html/
copying  tmp_2611.20785/./Figures/Heatmap_TPC_Results_K.png to _build/html/
copying  tmp_2611.20785/./Figures/Heatmap_TPC_Results_M.png to _build/html/
exported in  _build/html/2611.20785.md
    + _build/html/tmp_2611.20785/./Figures/radiustemperature

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\teff}{\ensuremath{T_{\mathrm{eff}} }}$
$\newcommand{\logg}{\ensuremath{{\mathrm{log}  } g }}$
$\newcommand{\feh}{\ensuremath{[{\mathrm{Fe/H}}] }}$
$\newcommand{\afe}{\ensuremath{[{\mathrm{\alpha/Fe}}] }}$
$\newcommand{\mk}{\ensuremath{\mathrm M_K }}$
$\newcommand{\msun}{\ensuremath{M_\odot }}$
$\newcommand{\Hii}{\ensuremath{\mathrm{H \textsc{ii}} }}$
$\newcommand{\Ha}{\ensuremath{\mathrm H_\alpha }}$
$\newcommand{\age}{\ensuremath{\tau}}$</div>



<div id="title">

# Model-Driven Stellar Spectroscopy with the Total Differential Spectral Model (TDSM): Framework and Its Application to 11 Million LAMOST Low-Resolution Spectra 

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.20633-b31b1b.svg)](https://arxiv.org/abs/2611.20633)<mark>Appeared on: 2026-11-17</mark> - 

</div>
<div id="authors">

L. Mou, et al. -- incl., <mark>M. Zhang</mark>

</div>
<div id="abstract">

**Abstract:** Deriving stellar astrophysical parameters from large-scale spectroscopic surveys requires accurate modeling of spectra across a wide wavelength coverage and a broad range of stellar parameter space, which remains a great challenge. In this work, we present \texttt{TDSM} , a physically motivated neural network architecture that self-consistently emulates both the flux and its first-order derivative of the ab initio stellar spectra. The method enables accurate and precise emulation of Kurucz synthetic spectra with severely blended features at $R \simeq 2000$ in a high-dimensional parameter space, achieving a typical pixel‑wise flux precision of 0.05 \% in the wavelength range $3700<\lambda<9000$ ${Å}$ over the parameter range of $3000 < T_{\rm eff} < 45,000$ K, $-1 < \log g < 5$ , and $-4.0 < \mathrm{[Fe/H]} < 0.5$ . The model is applied to LAMOST DR10 to derive stellar parameters for 11 million spectra spanning the full model parameter range.We conduct detailed refinement and calibration efforts to mitigate systematic errors, which are revealed to be significant and complex due to imperfections in the ab initio model atmosphere. The resultant catalog provides eight stellar labels -- $T_{\rm eff}$ , $\log g$ , $v_{\rm mic}$ , [ Fe/H ] , [ $\alpha$ /Fe ] , [ C/Fe ] , [ N/Fe ] , and [ Mg/Fe ] -- for 8 million unique stars (11 million spectra including repeat observations) across the full parameter space. This adds a valuable dataset for studies of Galactic structure and archaeology, stellar populations, and related areas of astrophysics.

</div>

<div id="div_fig1">

<img src="tmp_2611.20633/./calibration.png" alt="Fig9" width="100%"/>

**Figure 9. -**  Distributions of parameter differences between our estimates and the validation reference results. Results for dwarfs and giants are shown separately. The median trend is shown as a dashed line.  (*fig:calibration*)

</div>
<div id="div_fig2">

<img src="tmp_2611.20633/./example+loss.png" alt="Fig4" width="100%"/>

**Figure 4. -**  An overview of the \texttt{TDSM} model performance. The top panels show the \texttt{TDSM} spectrum emulation of a solar-like star, compared to the Kurucz model computation. The bottom-left panel shows the \texttt{TDSM} training loss and validation loss of both the flux values and their gradients, for a wavelength pixel as an example. The bottom-right panel presents the residual distribution of validation Kurucz spectra sample and the gradients. The overall scatter is $\sigma = 0.00053$ in flux residuals, and $\sigma = 0.000038$ in the gradients. (*fig:example+loss*)

</div>
<div id="div_fig3">

<img src="tmp_2611.20633/./self_validation.png" alt="Fig7" width="100%"/>

**Figure 7. -**   Self validation of the stellar-parameter determinations using synthetic test spectra. The horizontal axes show the true input parameters used in the Kurucz model calculation and spectral synthesis, while the vertical axes show the parameters recovered by \texttt{TDSM} fitting.
    Mean and standard deviation of the residuals are shown in the upper-left corner of each panel. For [N/Fe], metal-rich stars with \({\rm[Fe/H]}>-1\) are highlighted with red dots, and the standard deviation for both the overall sample and the metal-rich sample are presented.  (*fig:self_vali*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.20633"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand\arraystretch{1.8}$
$\newcommand\arraystretch{1.2}$
$\newcommand\arraystretch{1.2}$
$\newcommand\arraystretch{1.2}$</div>



<div id="title">

# Validation of Ariel candidates orbiting K-M stars

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.20785-b31b1b.svg)](https://arxiv.org/abs/2611.20785)<mark>Appeared on: 2026-11-17</mark> -  _Accepted for publication on Astronomy and Astrophysics_

</div>
<div id="authors">

G. Piccinini, et al. -- incl., <mark>I. J. M. Crossfield</mark>

</div>
<div id="abstract">

**Abstract:** One of the key goals of planetary population studies is to achieve broad coverage of the parameter space, which is not yet fully represented by confirmed planets. This limitation can be mitigated by including planets that have been statistically validated through a vetting procedure, even if they are not fully confirmed. In preparation for the Ariel mission, a Mission Candidate Sample has been assembled, comprising both confirmed planets and several unvalidated candidates. In this context, we apply a vetting analysis to a selected subsample of these candidates. Our methodology is based on the \texttt{TRICERATOPS} tool, aimed at confirming the presence of transiting planets or ruling out false-positive scenarios. We applied \texttt{TRICERATOPS} to a sample of 318 targets drawn from the last three Ariel TESS Planet Candidates catalogues. The analysis includes evaluation of statistical probabilities, as well as checks on stellar binarity indicators such as the Renormalized Unit Weight Error (RUWE) and the \texttt{IPD\_frac\_multi\_peak} values from $_ Gaia_$ Data Release 3. Our analysis validated 56 candidates as planets, identified 29 false positives, and classified 35 as likely planets. These results are intended to support the Ariel mission by providing a probabilistic priority ranking to guide follow-up observations and optimise targets for atmospheric characterisation studies.

</div>

<div id="div_fig1">

<img src="tmp_2611.20785/./Figures/radiustemperature.png" alt="Fig2" width="100%"/>

**Figure 2. -** Comparison between candidate planets and analysed sample. All elements from the last three TPC catalogues (green circles), the selected sample analysed in this work (blue stars), validated candidates including both newly validated targets and those previously validated in the literature (magenta circles), and targets that could not be analysed (yellow triangles). (*fig:radiustemp*)

</div>
<div id="div_fig2">

<img src="tmp_2611.20785/./Figures/Heatmap_TPC_Results_K.png" alt="Fig3" width="100%"/>

**Figure 3. -** Coverage of the full early-M and middle-K star sample compared to the three TPC catalogues. The total number of candidates (TPC) around middle-M and early-K stars (3500 $< T_{\rm eff} <$ 5000 K) is reported in black. The number of validated planets (VP) is green, the number of likely planets (LP) is orange, the number of false positives (FP) is red, and the number of Other (OT) is blue. Targets that were excluded from the validated sample following our checks (see Section \ref{validated}) are not included in the VP count. (*fig:Sample_KM*)

</div>
<div id="div_fig3">

<img src="tmp_2611.20785/./Figures/Heatmap_TPC_Results_M.png" alt="Fig4" width="100%"/>

**Figure 4. -** Coverage of the full M-dwarf star sample compared to the three TPC catalogues. The total number of candidates (TPC) orbiting M stars ($T_{\rm eff} <$ 3500 K) is reported in black. The number of validated planets (VP) is green, the number of likely planets (LP) is orange, the number of false positives (FP) is red, and the number of Other (OT) is blue. Targets that were excluded from the validated sample following our checks (see Section \ref{validated}) are not included in the VP count. (*fig:Sample_M*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.20785"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$</div>



<div id="title">

# Orbital decomposition and evolution of \texttt{LEGA-C}-like galaxies in \texttt{IllustrisTNG} from $z=1$ to $z=0$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.20884-b31b1b.svg)](https://arxiv.org/abs/2611.20884)<mark>Appeared on: 2026-11-17</mark> -  _Accepted for publication in A&A on 27/10/2026_

</div>
<div id="authors">

P. Kumar, et al. -- incl., <mark>A. Pillepich</mark>

</div>
<div id="abstract">

**Abstract:** Stellar orbits encode the dynamical history of galaxies and provide a physically motivated link between galaxy structures and assembly processes. Although orbit-based decompositions are well established in the Local Universe, their evolution across cosmic time, especially at higher redshifts, remains poorly explored. We investigate the orbital structure of \texttt{LEGA-C-like} galaxies in the \texttt{TNG100} simulation from $z=1$ to $z=0$ to identify how stellar mass, star formation activity, environment, merger history, gas content, and ex situ stellar mass growth shape their orbital transformation. We decomposed the stellar orbital structure of simulated galaxies into cold, warm, hot, and counter-rotating (CR) components using time-averaged orbital circularities. This provides a theoretical basis for future comparisons with Schwarzschild-based orbital decompositions of observed galaxies. We analyzed the galaxy populations at $z=1$ and $z=0$ , then tracked the descendants of individual $z=1$ galaxies to quantify their orbital evolution over this time. We find a strong mass dependence of orbital structure at $z=0$ as well as at $z=1$ . Galaxies below $\log(M_\star/M_\odot)\sim11.2$ have a combined warm and cold orbital fraction of $\sim70\%$ , whereas more massive systems have a hot-orbit fraction of $\sim60\%$ . These trends are consistent with assembly pathways driven predominantly by wet and dry major mergers, respectively. Tracking the same galaxies to $z=0$ reveals a systematic, but mass-dependent, evolution toward lower rotational support: At both low and high stellar masses, the hot-orbit fraction increases, whereas the relative change is greater in low-mass galaxies. The survival of cold structure depends strongly on the star formation state at $z=1$ ; star-forming progenitors retain substantially larger cold fractions than the quiescent progenitors. At both redshifts, the orbital fractions show a rise (decline) in hot and CR  (cold and warm) orbits with the ex situ stellar mass fraction at a fixed stellar mass. These results provide the first evolutionary framework for interpreting orbit-based decompositions of galaxies at the \texttt{LEGA-C} epoch. Stellar orbital fractions quantify the morphological and dynamical transformation of galaxies and show systematic evolutionary trends at the population level. Individual galaxies, however, follow diverse evolutionary pathways. Comparisons to future Schwarzschild-based decompositions of observed high-redshift galaxies can use these results to interpret the internal physical structure of galaxy populations and constrain theoretical models of galaxy evolution.

</div>

<div id="div_fig1">

<img src="tmp_2611.20884/./figs/orbital_fractions_combined.png" alt="Fig7" width="100%"/>

**Figure 7. -** Stellar orbital fractions within $2R_{1/2}$ as a function of stellar mass for \texttt{TNG100} galaxies at $z=1$(left) and $z=0$(right), for the \texttt{survived LEGA-C-like}(solid) and \texttt{whole/extended LEGA-C-like}(dashed) samples. The lines show the median orbital fractions, with shaded regions indicating the $1\sigma$ scatter. Top row: At both redshifts, orbital structure depends strongly on stellar mass: lower-mass galaxies are dominated by warm and cold orbits, whereas massive systems become dominated by hot orbits. From $z=1$ to $z=0$, the hot fraction increases, while the cold and warm components decline, indicating a systematic evolution toward dynamically hotter stellar structures. Middle row: Evolution is also shown to be dependent strongly on star formation activity: at fixed stellar mass, \texttt{quiescent} galaxies generally have higher (lower) hot (cold) fractions than \texttt{star-forming} galaxies, with the difference becoming particularly pronounced by $z=0$. Bottom row: In contrast, \texttt{central} and \texttt{satellite} galaxies follow broadly similar mass-dependent trends. (*fig:fraction_plot_z1_z0*)

</div>
<div id="div_fig2">

<img src="tmp_2611.20884/./figs/orbital_fractions_evolution.png" alt="Fig8" width="100%"/>

**Figure 8. -** Evolution of the stellar orbital fractions from $z=1$ to $z=0$ for all galaxies (top), star-forming galaxies (middle), and quiescent galaxies (bottom) binned by their $z=0$ stellar mass. The upper row shows the corresponding mean standard deviation values ($\sigma$) of the orbital fractions (see Appendix \ref{OrbitalEvolution_Stellar mass_Sigma}) . The orbital components show strongly mass dependent evolution: in the most massive bins, the hot (warm) component shows an increase (decline) from $z=1$ to $z=0$, whereas in the less massive bins, the warm components remain higher than the hot components across redshift. Meanwhile, the cold components continue to decline from $z=1$ to $z=0$, from their highest to their lowest fractions. Comparing star-forming and quiescent galaxies, we find that star-forming galaxies retain larger cold fractions, particularly at lower stellar masses. (*fig:orbital_evolution*)

</div>
<div id="div_fig3">

<img src="tmp_2611.20884/./figs/SFR_classification_evo_z_1_z_0.png" alt="Fig4" width="100%"/>

**Figure 4. -** \texttt{TNG100} galaxy populations from $z=1$ to $z=0$.
Middle panels: sSFR versus stellar mass at $z=1$(left) and $z=0$(right). Galaxies are classified relative to the SFMS as star-forming or quiescent. The solid blue line shows the median SFMS in 0.2 dex stellar-mass bins; galaxies with zero SFR are placed at $\mathrm{sSFR}=10^{-15},\mathrm{yr}^{-1}$ and marked with black stars.
Top panels: Stellar-mass distributions separated into \texttt{central} and \texttt{satellite} galaxies (top row) and star-forming and quiescent galaxies (second row). At $z=1$, the sample is dominated by star-forming centrals, while by $z=0$ quiescent galaxies and satellites are substantially more prevalent. The SFMS normalization also declines, with more galaxies lying below it. This transition is strongest among satellites, where quiescent systems dominate by $z=0$, consistent with environmental quenching after infall. The simultaneous growth of the quiescent central population, however, indicates that quenching also involves processes independent of environment.
 (*fig:star_formation*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.20884"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\hoggsay}[1]{{\color{red}[{\tiny Hogg say:} #1]}}$
$\newcommand{\psiang}{\psi}$
$\newcommand{\dmu}{\Delta\mu}$</div>



<div id="title">

# Gaia as a $0\farcs1$ point-source survey across the sky:: dynamical scene reconstruction from one‑dimensional scans

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.21510-b31b1b.svg)](https://arxiv.org/abs/2611.21510)<mark>Appeared on: 2026-11-17</mark> -  _Submitted to A&A. 17 pages, 8 figures. Comments are welcome. Code available at this https URL_

</div>
<div id="authors">

<mark>S. Shahaf</mark>, et al. -- incl., <mark>K. El-Badry</mark>, <mark>H.-W. Rix</mark>, <mark>R. Andrae</mark>

</div>
<div id="abstract">

**Abstract:** Gaia has observed every detected source on the sky 50 to 100 times, each time through a window of about 1 by 2 arcseconds at a different scan angle. The resulting one-dimensional flux profiles to be released in Gaia DR4 encode the two-dimensional arrangement of nearby point sources at ${\sim}$ 0.1 arcsecond resolution. Repeated observations also constrain their relative motion. We present a method to model the scene with a mixture of moving point sources. Complementing Gaia DPAC's source-environment reconstruction, our approach focuses on close, blended companions and explicitly models their relative motion across the observing baseline. We first fit each window with a few point sources and obtain their along-scan positions. We then triangulate these positions from different scan angles to identify candidate sources on the sky. A joint forward model of all windows measures positions, fluxes, relative proper motions, and parallax differences, including when the components are blended.In simulations, we recover companions at separations smaller than the instrumental profile width. For a moving companion at ${\sim} 75$ mas separation with a flux ratio of ${\sim} 0.2$ , we obtain a separation precision of ${\sim} 0.1$ mas and a relative proper-motion precision of ${\sim} 0.1$ mas yr $^{-1}$ per component, even in a scene "contaminated" by a third source. These precisions assume a known instrumental profile and independent Gaussian noise. We also recover differential parallax for a chance-aligned source. Modeling nearby contaminants prevents significant biases. With a Gaia-like instrumental profile, sampling and scan law, we show a sensitivity of ${\sim} 5$ mag in contrast at separations of $50$ to $150$ mas, and we discuss its dependence on the color term of Gaia's line-spread function. The method could extend the census of close companions and provide relative astrometry, and so dynamical information, along short orbital arcs. External measurements could also guide retrospective searches in Gaia's windows.

</div>

<div id="div_fig1">

<img src="tmp_2611.21510/./fig02_scan_to_skyline.png" alt="Fig2" width="100%"/>

**Figure 2. -** From along-scan detections to a two-dimensional source position. *(a)* Scan geometry for the simulated pair, with the primary at the origin (cross) and the companion marked by the star. Each line through the origin is the scan axis of one transit, and two transits are highlighted. The dotted segment drops the companion perpendicularly onto the scan axis, and the dot marks the along-scan position measured from the window. Figure \ref{fig:per_window_flow} shows how a single window is reduced to such along-scan positions. *(b)* Along-scan detections as a function of scan angle, with formal positional uncertainties. The dashed curve is the injected track, $\rho\cos(\psi-\varphi)$. The gray band marks projected separations for which the noise-free blended profile of this pair, with a flux ratio of $0.2$, has a single peak. This is not a boundary for detecting a second source. *(c)* The triangulation map combines bands around the sky lines allowed by the detections, with widths set by their positional uncertainties. The common intersection identifies a candidate companion position. The inset shows the subsequent joint window fit described in Section \ref{sec:bp-measure}, with profiled $\Delta\chi^2$ contours corresponding to $1$, $2$, and $3\sigma$. The circle marks the map-associated starting position, the cross the maximum-likelihood position, and the star the injected position. (*fig:scan_to_skyline*)

</div>
<div id="div_fig2">

<img src="tmp_2611.21510/./fig06_detection_plane.png" alt="Fig6" width="100%"/>

**Figure 6. -** Detection limit, positional uncertainty, and color bias of a companion in the plane of separation and flux ratio $\mathcal{S}$, next to a primary of $G=14$(a), $16$(b), and $18$(c). The primary is a K star with $\nu_{\rm eff}=1.42 \mu{\rm m}^{-1}$, observed in $47$ transits with the noise of equation \eqref{eq:noise} at a background of $b=100$ electrons, and modeled with the LSF digitized from [Rowell, Davidson and Lindegren (2021)](). The red band spans the flux ratios at which $90$ to $95\%$ of campaigns detect the companion at a false-detection rate of at most $1\%$, with the color of the LSF known. The dashed curve is the $95\%$ limit with the threshold recalibrated for a color error of $|\Delta\nu|=0.03 \mu{\rm m}^{-1}$. Black contours show the quoted positional uncertainty of the companion. The dotted lines mark where the noise-free bias of the companion's position, for a model color in error by $|\Delta\nu|=0.007 \mu{\rm m}^{-1}$(Section \ref{sec:lsf-errors}), equals one and three times that uncertainty, and the bias is larger on the side of closer and fainter companions. The lines are not drawn below the band, and in panel (c) the bias stays below the uncertainty. The triangle on the right axis marks the flux ratio of a companion of $G=21$. (*fig:detection-plane*)

</div>
<div id="div_fig3">

<img src="tmp_2611.21510/./fig03_dynamics.png" alt="Fig3" width="100%"/>

**Figure 3. -** Motion compensation for a bound companion (a) and a chance-aligned background source (b). Both scenes have a separation of $200$ mas at the reference epoch, a flux ratio of $0.3$, and the uniform noise of Section \ref{sec:static-example}, sampled at the same $47$ GOST-predicted transits as Figure \ref{fig:scan_to_skyline}. *Top:* along-scan detections before compensation (open circles) and after removing the fitted relative motion and, for the background source, differential parallax (filled points). Crosses mark detections rejected by pooling. The solid curve shows the fitted scene at the reference epoch. The compensation uses the parameters from the joint window fit. *Bottom:* triangulation maps before and after compensation for each scene. Stars mark the injected position at the reference epoch. Accounting for motion brings the projected detections into closer agreement and concentrates the map near that position. (*fig:dynamics*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.21510"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

155  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

20  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

6  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
